# MLflow Experiment Tracking with Git SHA Integration


This notebook was created to design, train, and track logistic regression experiments on a cholesterol dataset using **MLflow**.  
Our plan is to build a reproducible workflow that covers **data cleaning, feature engineering, model training, evaluation, and experiment logging**.  
By integrating MLflow, we ensure that each run is versioned, tagged with metadata (release info, owner, framework, commit SHA), and stored with artifacts for traceability.  
The goal is to provide a **modular, recruiter‑friendly, and production‑ready pipeline** that demonstrates end‑to‑end MLOps practices, highlights reproducibility, and supports future hyperparameter tuning or model comparisons.


In [80]:
import os
import pandas as pd
import numpy as np
import subprocess
import hashlib
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc,precision_score,recall_score
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score,f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
import argparse
import warnings
import mlflow
import mlflow.sklearn
from mlflow.models.signature import ModelSignature, infer_signature
from mlflow.types.schema import Schema,ColSpec
from mlflow.tracking import MlflowClient
import mlflow.xgboost
#import pickle
import pathlib
warnings.filterwarnings("ignore", category=UserWarning)
import sys
import sys
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

In [81]:
def get_git_commit_sha() -> str:
    """Return current Git commit SHA, or 'unknown' if unavailable."""
    try:
        sha = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], stderr=subprocess.STDOUT
        )
        return sha.decode("utf-8").strip()
    except Exception:
        return "unknown"

In [82]:
def generate_commit_sha(author="Abhishek <abhishek@example.com>", message="Commit"):
    """
    Return the actual Git commit SHA if available,
    otherwise fall back to a synthetic SHA.
    """
    sha = get_git_commit_sha()
    if sha != "unknown":
        return sha

    # Fallback: synthetic SHA
    commit_content = f"""tree 4b825dc642cb6eb9a060e54bf8d69288fbee4904
author {author} {int(time.time())} +0000
committer {author} {int(time.time())} +0000

{message}
"""
    header = f"commit {len(commit_content)}\0"
    store = header + commit_content
    return hashlib.sha1(store.encode("utf-8")).hexdigest()

In [83]:
def get_your_file_path():
    # Get current working directory
    cwd = pathlib.Path().resolve()   # resolves to notebook’s folder
    filename = "hurt_cholesterol.csv"
    file_path = cwd / filename

    # Check if file exists
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    return file_path,filename

API provides a reliable way to attach a Git commit `SHA` to your MLflow runs for `reproducibility`. The function `get_git_commit_sha()` tries to fetch the current commit hash using `git rev-parse HEAD`; if Git isn’t available, it returns `"unknown"`. The wrapper `generate_commit_sha()` first calls that function—if a real SHA is found, it uses it directly. If not, it builds a synthetic commit `SHA` by constructing a pseudo‑commit object with a fixed tree ID, author, timestamp, and message, then hashing it with SHA‑1 to mimic Git’s internal storage format. In practice, this means your experiments always get a commit identifier: either the actual Git hash when the repo is available, or a deterministic synthetic hash when it isn’t. This ensures every MLflow run can be traced back to a `code version, strengthening reproducibility, auditability, and code‑to‑model traceability` in MLOps pipeline.

In [84]:
# Function: drop rows with non-numeric values
def drop_non_numeric(df_frame):
    cleaned_df = df_frame.copy()
    for col in cleaned_df.columns:
        # Try to convert column to numeric
        cleaned_df[col] = pd.to_numeric(cleaned_df[col], errors='coerce')
    # Drop rows where conversion failed (NaN introduced)
    cleaned_df = cleaned_df.dropna()
    return cleaned_df

In [85]:
def categorize_chol(val):
    if val < 200:
        return "Normal", 0
    elif 200 <= val <= 239:
        return "Medium", 1
    else:
        return "High", 1

In [86]:
# evaluation function
def eval_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    return rmse, mae, r2

In [87]:
def get_cleaned_data():
    # Get current working directory    
    #cwd = os.path.dirname(os.path.abspath(__file__))
    # Specify dataset filename
    file_path,filename = get_your_file_path()
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ Dataset not found at: {file_path}")
    df = pd.read_csv(file_path)
    null_counts = df.isnull().sum()
    #if null_counts.sum() > 0:
    #    print("\n✅ Null values are present in the dataset.")
    #else:
    #    print("\n❌ No null values found in the dataset.")
    # Apply cleaning
    df_clean = drop_non_numeric(df)
    #print("Cleaned shape:", df_clean.shape)
    df_clean[['chol_category_label', 'chol_category_code']] = df_clean['chol'].apply(
        lambda x: pd.Series(categorize_chol(x))
        )
    # Define columns and target
    columns = ['age', 'sex', 'cp', 'trestbps', 'fbs', 'restecg', 'thalach', 'exang',
           'oldpeak', 'slope', 'ca', 'thal', 'num', 'chol_category_code']
    target = "chol_category_code"
    binary_vars = ['sex', 'fbs', 'exang']
    # Separate features and target
    X = df_clean[columns].drop(columns=[target])
    y = df_clean[target]
    
    # Identify numeric columns to scale (exclude binary + target)
    numeric_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
    numeric_cols = [col for col in numeric_cols if col not in binary_vars]
    
    # Apply StandardScaler
    scaler = StandardScaler()
    X_scaled = X.copy()
    X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])
    
    # Store result in df_scale (features + target)
    df_scale = X_scaled.copy()
    df_scale[target] = y
    
    #print("Scaled dataset preview:")
    #print(df_scale.columns)
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42, stratify=y
    )
    #print("\nTrain set shape:", X_train.shape, y_train.shape)
    #print("Test set shape:", X_test.shape, y_test.shape)
    return X_train, X_test, y_train, y_test

The `get_cleaned_data()` function loads the dataset from a specified path, checks for missing values, drops non‑numeric rows, and categorizes cholesterol into labels and codes; it then selects relevant features while marking binary variables, separates predictors (`X`) and target (`y`), scales numeric columns using `StandardScaler`, recombines the scaled features with the target, and finally performs a stratified 80/20 train‑test split to return `X_train, X_test, y_train, y_test` for model training and evaluation.

In [88]:
def logistic_model_1(X_train, X_test, y_train, y_test):
    # Logistic Regression with elasticnet penalty, saga solver
    # 1. saga + elasticnet + l1_ratio=0.1, C=10
    C_value = 10
    L_1_ratio = 0.1
    Solver = 'saga'
    Panality = 'elasticnet'
    model = LogisticRegression(
    penalty=Panality,
    solver=Solver,
    l1_ratio=L_1_ratio,
    C=C_value,
    max_iter=10000
    )
    model.fit(X_train, y_train)

    # Predictions
    y_pred_prob = model.predict_proba(X_test)[:, 1]  # probability of class 1
    y_pred_class = model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred_class)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    mse = mean_squared_error(y_test, y_pred_prob)
    mae = mean_absolute_error(y_test, y_pred_prob)

    # Store results in DataFrame
    results_df = pd.DataFrame([{
        "C": C_value,
        "l1_ratio": L_1_ratio,
        'solver': Solver,
        'penalty' : Panality,
        "Accuracy": acc,
        "ROC_AUC": roc_auc,
        "MSE": mse,
        "MAE": mae
    }])
    print("================ Logistice Model ================")
    print(f'Logistice accureacy {acc}')
    return model, results_df

The `logistic_model_1()` function trains a logistic regression model with an `elasticnet penalty` using the saga solver `(C=10, l1_ratio=0.1)`, fits it on the training data, predicts both class probabilities and labels on the test set, evaluates performance with metrics such as accuracy, ROC‑AUC, MSE, and MAE, stores these results along with model parameters in a DataFrame, prints a summary of the accuracy, and finally returns the trained model together with the results DataFrame for further analysis.

In [89]:
def get_uri_path() -> str:
    """
    Return MLflow tracking URI path based on current working directory.
    Ensures 'mlruns' and '.trash' folders exist.
    """
    cwd = os.getcwd()
    mlruns_path = os.path.join(cwd, "mlruns")
    trash_path = os.path.join(mlruns_path, ".trash")

    # Create both directories if missing
    os.makedirs(mlruns_path, exist_ok=True)
    os.makedirs(trash_path, exist_ok=True)

    return f"file:{mlruns_path}"


In [ ]:
def run_experiment(uri='default_path', experiment_name='default_exp', model_name='logistic_model',
                   run_name='default_run', model_func=None, model_args=None, tags=None, commit_sha=None):
    # Set tracking directory explicitly
    mlflow.set_tracking_uri(uri)
    print("The set tracking uri is ", mlflow.get_tracking_uri())
    
    # ✅ Use set_experiment to avoid duplicate errors
    exp = mlflow.set_experiment(experiment_name=experiment_name)
    exp_id = exp.experiment_id
    get_exp = mlflow.get_experiment(exp_id)
    print("Name:", get_exp.name)
    print("Experiment_id:", get_exp.experiment_id)
    print("Artifact Location:", get_exp.artifact_location)
    print("Tags:", get_exp.tags)
    print("Lifecycle_stage:", get_exp.lifecycle_stage)
    print("Creation timestamp:", get_exp.creation_time)

    with mlflow.start_run(experiment_id=exp_id, run_name=run_name):
        # Set tags if provided
        if tags:
            mlflow.set_tags(tags)

        # ✅ Only set commit SHA tag if provided
        if commit_sha is not None:
            mlflow.set_tag("git_commit", commit_sha)
            print(f"Tagged MLflow run with commit SHA: {commit_sha}")

        # 🔑 Log environment spec and Docker digest
        mlflow.log_artifact("environment.yml")   # Conda environment file
        #mlflow.set_tag("docker_image_digest", "<sha256:...>")  # Replace with actual digest

        # Train and evaluate model
        model, results_df = model_func(**model_args)

        # Log parameters
        for param in ["C", "l1_ratio", "solver", "penalty"]:
            if param in results_df.columns:
                mlflow.log_param(param, results_df.loc[0, param])

        # Log metrics
        for metric in ["Accuracy", "ROC_AUC", "MSE", "MAE"]:
            if metric in results_df.columns:
                mlflow.log_metric(metric, results_df.loc[0, metric])

        # Log model
        mlflow.sklearn.log_model(model, name=model_name, serialization_format="skops")

        # Log artifacts (optional)
        mlflow.log_artifacts("Logistice_Regressions_Models/")
        
        # Print artifact URI
        artifacts_uri = mlflow.get_artifact_uri()
        print("The artifact path is", artifacts_uri)

    mlflow.end_run()

    # Show last run info
    run = mlflow.last_active_run()
    if run:
        print("Active run id:", run.info.run_id)
        print("Active run name:", run.info.run_name)

        # 🔑 Register model in MLflow Model Registry
        print(' MLflow Model Registry')
        mlflow.register_model(
            model_uri=f"runs:/{run.info.run_id}/{model_name}",
            name="BinaryClassifications"
        )

        # ✅ Dynamically fetch latest version
        client = MlflowClient()
        latest_versions = client.get_latest_versions("BinaryClassifications")
        latest_version = latest_versions[0].version
        print(f"Loading latest model version: {latest_version}")

        relod_model = mlflow.pyfunc.load_model(model_uri=f"models:/BinaryClassifications/{latest_version}")
        predicted_qualities = relod_model.predict(model_args['X_test'])
        (rmse, mae, r2) = eval_metrics(model_args['y_test'], predicted_qualities)

        print("  RMSE_test: %s" % rmse)
        print("  MAE_test: %s" % mae)
        print("  R2_test: %s" % r2)


The `run_experiment()` function manages a complete MLflow workflow by setting the `tracking URI, creating or retrieving an experiment`, and starting a run with optional tags; if a `commit_sha` is provided it attaches it as a `reproducibility tag`, then trains and evaluates a model via the supplied model_func, logs parameters and metrics from the results DataFrame, saves the model and `artifacts`, prints the `artifact path`, and after ending the run it retrieves the last active run, `registers` the model in the `MLflow Model Registry`, `dynamically` loads the latest version, performs predictions on the test set, `evaluates` them with RMSE, MAE, and R², and prints these results, ultimately `ensuring experiment tracking`, `model versioning`, and `reproducibility` are all integrated.

In [91]:
def main():
    warnings.filterwarnings("ignore")
    np.random.seed(40)
    # Load data
    X_train, X_test, y_train, y_test = get_cleaned_data()
    # Example usage
    tags = {
    "Work": "Embedded Platform",
    "release.candidate": "RELAY_01",
    "release.version": "1.0.10",
    "dataset": "Cholesterol",
    "experiment.stage": "hyperparameter_tuning",
    "owner": "abhishek",
    "framework": "scikit-learn",
    "model.type": "logistic"
    }
    #uri_path = get_uri_path()   
    uri_path='http://127.0.0.1:5000'
    run_experiment(uri=uri_path,
                   experiment_name='logistice_l1', 
                   model_name='logistice_regressions',
                   run_name='Model_runs', 
                   model_func=logistic_model_1,
                   model_args={
                                "X_train": X_train,
                                "X_test": X_test,
                                "y_train": y_train,
                                "y_test": y_test,
        
                                },
                   tags=tags,
                   commit_sha=None)
    
    #========================================================
    tags = {
    "Work": "Embedded Platform",
    "release.candidate": "RELAY_02",
    "release.version": "1.0.11",
    "dataset": "Cholesterol",
    "experiment.stage": "hyperparameter_tuning",
    "owner": "abhishek",
    "framework": "scikit-learn",
    "model.type": "logistic_sha"
    }
    #uri_path = get_uri_path()   
    uri_path='http://127.0.0.1:5000'
    run_experiment(uri=uri_path,
                   experiment_name='logistice_sha', 
                   model_name='logistice_reg_git_sha',
                   run_name='Model_git_sha', 
                   model_func=logistic_model_1,
                   model_args={
                                "X_train": X_train,
                                "X_test": X_test,
                                "y_train": y_train,
                                "y_test": y_test,
        
                                },
                   tags=tags,
                   commit_sha=generate_commit_sha())

The `main()` function orchestrates the full experiment workflow by first suppressing warnings and fixing the random seed, then loading the cleaned dataset via `get_cleaned_data()` to obtain `training` and `test` splits; it defines experiment tags with metadata such as work context, `release version, dataset, stage, owner, framework, and model type, sets the MLflow tracking URI, and calls run_experiment()` twice—first with `commit_sha=None` for a standard run (`logistice_l1`), and then with a dynamically generated `commit SHA` for a reproducible run (`logistice_sha`)—ensuring both experiments are logged with `parameters, metrics, and artifacts` while maintaining version control and traceability.

In [92]:
if __name__ == "__main__":
    main()

The set tracking uri is  http://127.0.0.1:5000
Name: logistice_l1
Experiment_id: 12
Artifact Location: file:C:/Users/abhis/mlruns/12
Tags: {}
Lifecycle_stage: active
Creation timestamp: 1780403095622
================ Logistice Model ================
Logistice accureacy 0.8333333333333334


Registered model 'BinaryClassifications' already exists. Creating a new version of this model...
2026/06/02 18:18:30 WARNING mlflow.tracking._model_registry.fluent: Run with id d7f94e66d12645cdb4c931dd7fd0ac31 has no artifacts at artifact path 'logistice_regressions', registering model based on models:/m-5cc178fe2eba486e8bb8a7c77c733a93 instead


The artifact path is file:C:/Users/abhis/mlruns/12/d7f94e66d12645cdb4c931dd7fd0ac31/artifacts
🏃 View run Model_runs at: http://127.0.0.1:5000/#/experiments/12/runs/d7f94e66d12645cdb4c931dd7fd0ac31
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/12
Active run id: d7f94e66d12645cdb4c931dd7fd0ac31
Active run name: Model_runs
 MLflow Model Registry


2026/06/02 18:18:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: BinaryClassifications, version 4
Created version '4' of model 'BinaryClassifications'.
2026/06/02 18:18:30 INFO mlflow.tracking.fluent: Experiment with name 'logistice_sha' does not exist. Creating a new experiment.


Loading latest model version: 4
  RMSE_test: 0.408248290463863
  MAE_test: 0.16666666666666666
  R2_test: -0.19999999999999996
The set tracking uri is  http://127.0.0.1:5000
Name: logistice_sha
Experiment_id: 13
Artifact Location: file:C:/Users/abhis/mlruns/13
Tags: {}
Lifecycle_stage: active
Creation timestamp: 1780404510350
Tagged MLflow run with commit SHA: 0bd7de1743e30ee3e515c3e3050d6b358a855faa
================ Logistice Model ================
Logistice accureacy 0.8333333333333334


Registered model 'BinaryClassifications' already exists. Creating a new version of this model...
2026/06/02 18:18:39 WARNING mlflow.tracking._model_registry.fluent: Run with id 2932bcbb870b434889aac1640d7032aa has no artifacts at artifact path 'logistice_reg_git_sha', registering model based on models:/m-0fd17de035c44946b99e5ae035031630 instead
2026/06/02 18:18:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: BinaryClassifications, version 5


The artifact path is file:C:/Users/abhis/mlruns/13/2932bcbb870b434889aac1640d7032aa/artifacts
🏃 View run Model_git_sha at: http://127.0.0.1:5000/#/experiments/13/runs/2932bcbb870b434889aac1640d7032aa
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/13
Active run id: 2932bcbb870b434889aac1640d7032aa
Active run name: Model_git_sha
 MLflow Model Registry


Created version '5' of model 'BinaryClassifications'.


Loading latest model version: 5
  RMSE_test: 0.408248290463863
  MAE_test: 0.16666666666666666
  R2_test: -0.19999999999999996


This workflow demonstrates disciplined MLOps practice:
- **Experiment Tracking:** Each run is logged with parameters, metrics, and artifacts.  
- **Version Control:** Models are registered in MLflow’s Model Registry, automatically creating new versions (v4, v5).  
- **Reproducibility:** One run is explicitly tagged with a Git commit SHA, ensuring the model can be traced back to the exact code version.  
- **Evaluation:** Metrics like Accuracy (83%), RMSE, MAE, and R² are logged, showing both classification performance and regression‑style error analysis.  

I implemented MLflow experiment tracking and model registry integration. Each logistic regression run was logged with parameters, metrics, and artifacts, and tied to a Git commit SHA for reproducibility. Models were versioned automatically in the registry (v4, v5), achieving ~83% accuracy.

![Model sha-1](image.png)


**Metrics & Parameters – Model_git_sha:**

Accuracy ~83% → The model correctly classifies most cases. ROC_AUC ~0.50 → This is weak; the model is barely better than random guessing in terms of ranking positive vs. negative cases.MSE / MAE ~0.15 and ~0.27 → These error values show the average deviation between predicted probabilities and actual labels. Parameters Logistic regression with C=10, l1_ratio=0.1, solver=saga, penalty=elasticnet. This highlights hyperparameter tuning with elastic net regularization, balancing L1 and L2 penalties.

![run_id with tags](image-1.png)

**Experiment Metadata – logistice_sha:**

#### How this helps with code versioning:

**Git Commit Tagging:**  

Each run is tagged with the exact Git commit SHA (0bd7de1743e30ee3e515c3e3050d6b358a855faa). This means you can always trace a model back to the precise code snapshot that produced it. No ambiguity about which version of the notebook or script was used.

**Experiment Registry Integration:**  

When MLflow registers a new model version (v4, v5, etc.), it links that version to the run metadata. Because the run carries the commit SHA, the model registry now implicitly ties each model version to a code version.

**Reproducibility:**  

If someone asks “Which code produced model v5?” you can look at the run tags and immediately know the Git commit. You can then check out that commit in Git and reproduce the exact training run.

**Auditability:**

In production environments, this is critical. If a model misbehaves, you can roll back to the commit that trained the last stable version. This closes the loop between source control (Git) and experiment tracking (MLflow).